# 第5章 Notebook：乗降の最小セルオートマトン

対応章: [`../chapters/05_agent_boarding_basic.md`](../chapters/05_agent_boarding_basic.md)

この notebook は、卒業研究準備セミナーの数値実験用である。上から順に実行すれば、本文で説明した図を再現できる。設定パラメータは上部のセルにまとめてある。乱数は seed を固定している。

## 1. ライブラリ読み込み

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

## 2. パラメータ設定

In [ ]:
H, W = 11, 21          # grid (train | door | platform)
DOOR_COL = 10          # wall column with a door
DOOR_HALF = 1          # half door width -> 3 open rows
N_ALIGHT, N_BOARD = 14, 14
MAX_STEPS = 600

## 3-6. グリッド・エージェント・移動ルール・衝突回避

降車客は右（ホーム端）へ、乗車客は左（車内端）へ向かう。1 セル 1 人で衝突を回避し、逐次更新（毎ステップ順番をシャッフル）する。ドア中央列は常に開けておく。

In [ ]:
def make_grid(H=11, W=21, door_col=10, door_half=1):
    passable = np.ones((H, W), bool)
    cy = H // 2
    wall = np.ones(H, bool)
    wall[cy - door_half: cy + door_half + 1] = False
    passable[wall, door_col] = False   # wall except door rows
    return passable, door_col, cy

def place_agents(passable, door_col, n_alight, n_board, seed):
    rng = np.random.default_rng(seed)
    H, W = passable.shape
    occ = np.zeros((H, W), bool)
    train = [(r, c) for r in range(H) for c in range(door_col) if passable[r, c]]
    plat = [(r, c) for r in range(H) for c in range(door_col + 1, W) if passable[r, c]]
    rng.shuffle(train); rng.shuffle(plat)
    agents = []
    for (r, c) in train[:n_alight]:
        agents.append({'pos': (r, c), 'type': 'alight', 'done': False}); occ[r, c] = True
    for (r, c) in plat[:n_board]:
        agents.append({'pos': (r, c), 'type': 'board', 'done': False}); occ[r, c] = True
    return agents, occ

def simulate(H=11, W=21, door_col=10, door_half=1, n_alight=14, n_board=14,
             standers=0, stander_yield=0.0, max_steps=600, seed=0):
    rng = np.random.default_rng(seed)
    passable, dc, cy = make_grid(H, W, door_col, door_half)
    agents, occ = place_agents(passable, dc, n_alight, n_board, seed)
    # fixed standers just outside the door; keep the centre door row open
    cand = [(cy + 1, dc + 1), (cy - 1, dc + 1), (cy, dc + 2),
            (cy + 1, dc + 2), (cy - 1, dc + 2)]
    stander_set = set()
    for (r, c) in cand:
        if len(stander_set) >= standers:
            break
        if 0 <= r < H and 0 <= c < W and passable[r, c] and not occ[r, c]:
            stander_set.add((r, c))

    def blocked(nr, nc):
        if occ[nr, nc]:
            return True
        if (nr, nc) in stander_set:
            return rng.random() > stander_yield   # fixed (yield=0) always blocks
        return False

    density = np.zeros((H, W))
    steps = 0
    while steps < max_steps and any(not a['done'] for a in agents):
        order = [a for a in agents if not a['done']]
        rng.shuffle(order)
        for a in order:
            r, c = a['pos']
            tc = W - 1 if a['type'] == 'alight' else 0
            cur = (abs(c - tc), abs(r - cy))
            best = None
            for dr, dcc in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                nr, nc = r + dr, c + dcc
                if 0 <= nr < H and 0 <= nc < W and passable[nr, nc] and not blocked(nr, nc):
                    sc = (abs(nc - tc), abs(nr - cy))
                    if sc < cur and (best is None or sc < best[0]):
                        best = (sc, nr, nc)
            if best is not None and best[1:] not in stander_set:
                nr, nc = best[1], best[2]
                occ[r, c] = False; occ[nr, nc] = True; a['pos'] = (nr, nc)
                r, c = nr, nc
            if a['type'] == 'alight' and c == W - 1:
                a['done'] = True; occ[r, c] = False
            elif a['type'] == 'board' and c == 0:
                a['done'] = True; occ[r, c] = False
        density += occ
        steps += 1
    return steps, density, passable

## 7. シミュレーション実行：障害物なし vs ドア付近に固定障害物

In [ ]:
steps0, dens0, passable = simulate(H, W, DOOR_COL, DOOR_HALF, N_ALIGHT, N_BOARD,
                                   standers=0, max_steps=MAX_STEPS, seed=0)
steps1, dens1, _ = simulate(H, W, DOOR_COL, DOOR_HALF, N_ALIGHT, N_BOARD,
                            standers=2, stander_yield=0.0, max_steps=MAX_STEPS, seed=0)
print('dwell steps (no obstacle):', steps0)
print('dwell steps (2 fixed standers):', steps1)

## 8. 完了時間と密度分布の可視化

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
im0 = ax[0].imshow(dens0, cmap='viridis'); ax[0].set_title(f'density, no obstacle (T={steps0})')
ax[0].axvline(DOOR_COL, color='w', ls='--', lw=1); fig.colorbar(im0, ax=ax[0], fraction=0.046)
im1 = ax[1].imshow(dens1, cmap='viridis'); ax[1].set_title(f'density, 2 standers (T={steps1})')
ax[1].axvline(DOOR_COL, color='w', ls='--', lw=1); fig.colorbar(im1, ax=ax[1], fraction=0.046)
plt.tight_layout(); plt.show()

## 9. 乗車人数を変えた停車時間

In [ ]:
ns = [6, 10, 14, 18, 22]
dwell = [simulate(H, W, DOOR_COL, DOOR_HALF, n, n, standers=0,
                  max_steps=MAX_STEPS, seed=0)[0] for n in ns]
plt.figure(figsize=(6, 4))
plt.plot(ns, dwell, 'o-'); plt.xlabel('alight = board count'); plt.ylabel('dwell steps')
plt.title('dwell time vs crowd size'); plt.tight_layout(); plt.show()

## 10. 課題（自分で変更する）

1. `DOOR_HALF` を 0 にしてドアを狭くすると停車時間はどう変わるか。
2. 乗車人数だけ増やしたとき、停車時間は線形に増えるか急増するか調べよ。

In [ ]:
# === 課題セル ===
for dh in [0, 1, 2]:
    t = simulate(H, W, DOOR_COL, dh, N_ALIGHT, N_BOARD, standers=0, max_steps=MAX_STEPS, seed=0)[0]
    print(f'door_half={dh}: dwell={t}')